# LA 02b — First-touch versus second-touch aggressive flow and zone volume

**Research-only causal feature enrichment; no trade P&L.** This module *does not select new signals*. It uses the 452 price-only A02 candidates and reconstructs trade flow during each actual touch candle, plus historical price-volume information available before the first touch. Trading rules, expected target/stop probabilities, fee assumptions and thresholds remain **unfrozen**.

The local Binance USD-M BTC/ETH aggregated trade files are scanned in bounded batches. Data are not written back to the raw archive. Input events are loaded from `02/02_price_only_candidates.csv`, and results are written to `02b/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter in repository or a subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a02b import (
    Config, LA_A02B_VERSION, run_a02b,
)
assert LA_A02B_VERSION == "LA-A02b-0.1.0", "Pull repository and restart kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "02b"
CFG = Config()
print("Version:", LA_A02B_VERSION, "| Root:", DATA_ROOT, "| Output:", OUT)
print("Mode: EXPLORATORY FEATURE-ONLY; no forward outcomes / P&L")

In [ ]:
decision = run_a02b(DATA_ROOT, REPO, OUT, cfg=CFG)
decision

In [ ]:
features = pd.read_csv(OUT / "02b_touch_features.csv")
coverage = pd.read_csv(OUT / "02b_coverage.csv")
display(coverage)
print("Events:", len(features), "| Presence:", features.touch_status.value_counts(dropna=False).to_dict())
assert len(features) == decision["source_a02_candidate_count"], "A02 source rows changed"
assert features.a02_candidate_row.is_unique, "Duplicated candidate identity"
for label, cols in [
    ("touch observations", ["symbol", "side", "first_touch_close", "second_touch_close", "first_zone_quote", "second_zone_quote", "zone_quote_touch_ratio", "first_opposing_zone_share", "second_opposing_zone_share"]),
    ("relative to time around level", ["symbol", "side", "intersecting_1m_candles", "early_zone_quote_per_intersecting_1m", "late_zone_quote_per_intersecting_1m", "late_early_zone_rate_ratio", "zone_active_print_seconds_between_touches"]),
    ("pre-first-touch profile", ["symbol", "level_price", "pre_first_profile_status", "pre_first_poc_price", "pre_first_poc_distance_bp", "pre_first_profile_zone_share"]),
]:
    print("\n",label)
    display(features[cols].head(10))
print("\nMissing ratio counts:")
print(features[["zone_quote_touch_ratio", "late_early_zone_rate_ratio", "pre_first_poc_price"]].isna().sum())

## Interpretation before LA 03

1. **First/second touches** are exactly the completed 1-minute rejection candles already selected in A02, and the order flow uses only trades strictly before the signal timestamp. No retroactive stop changes or entries have been modeled.
2. **Near-level volume** uses the *actual aggregated trade price* within `[level − halfwidth, level + halfwidth]`, not 1bp midpoint bins. This is only executed volume, not passive order-book depth or confirmed iceberg activity.
3. **Dwell** is measured as the count of 1-minute candles whose OHLC intersects the zone. It is a price-visitation proxy, not seconds actually spent at that price. Separate “active print seconds” count seconds with a trade **at the level**, not continuous market occupation.
4. **First versus second volume** is normalized both per touch candle (fixed duration) and as a fraction of each full minute's executed quote volume. Time-between-touch early/late intensity is divided by the count of minute bars intersecting the zone, not by the entire elapsed gap.
5. **Zero first-touch volume** means the growth ratio remains undefined (`NaN`), rather than being replaced by zero or infinity. Read `touch_status` and `pre_first_profile_status` separately. Files missing before first touch do not erase available two-touch features.
6. The **pre-first point of control (POC)** is built from an untuned trailing 30-minute profile ending *before the first rejection candle starts*. It is only a reference attribute; A02b does **not** create events at POC or low-volume nodes.
7. Look at both first and second candle `*_trade_kline_quote_error` columns; large errors suggest a trade-archive completeness or alignment issue.

**Next:** commit the executed notebook plus all files in `notebooks/Liquidity_Absorption/02b/`, then request a review. A03 requires a separate, owner-reviewed specification for executable entry, stop/BE/trailing timing, causal profit-target discovery, and Binance-specific trading costs.